1. 상품 url 수집 함수

In [1]:
import time
import json
import random
import requests
import pandas as pd

from selenium import webdriver
from selenium.webdriver.chrome.options import Options


# =========================
# 1. Selenium으로 쿠키 받아오기
# =========================

def get_local_driver():
    options = Options()
    options.add_argument("--start-maximized")
    driver = webdriver.Chrome(options=options)
    return driver


def make_ohouse_session():
    driver = get_local_driver()

    driver.get("https://contents.ohou.se/projects/185117")
    time.sleep(5)

    cookies = driver.get_cookies()
    driver.quit()

    session = requests.Session()

    for cookie in cookies:
        session.cookies.set(
            cookie["name"],
            cookie["value"],
            domain=cookie.get("domain")
        )

    headers = {
        "accept": "application/json",
        "content-type": "application/json",
        "referer": "https://contents.ohou.se/projects/185117?affect_type=ProjectSelfIndex&affect_id=1",
        "ohouse-trust-client": "ohs-web-client",
        "user-agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/147.0.0.0 Safari/537.36",
    }

    return session, headers


# =========================
# 2. API 호출
# =========================

def fetch_project_api(content_id, session, headers):
    api_url = f"https://contents.ohou.se/api/projects/{content_id}"

    res = session.get(api_url, headers=headers)

    if res.status_code != 200:
        print(f"❌ {content_id} 실패: {res.status_code}")
        return None

    try:
        return res.json()
    except:
        print(f"❌ {content_id} JSON 변환 실패")
        return None


# =========================
# 3. 게시글 기본정보 추출
# =========================

def parse_post_info(data, content_id):
    d = data.get("data", {})

    writer = d.get("writer", {}) or {}
    features = d.get("features", {}) or {}

    post_row = {
        "contentId": d.get("contentId", content_id),
        "contentType": d.get("contentType"),
        "title": d.get("title"),
        "createdAt": d.get("createdAt"),
        "updatedAt": d.get("updatedAt"),
        "url": f"https://contents.ohou.se/projects/{content_id}",
        "coverImageUrl": d.get("coverImageUrl"),

        "viewCount": d.get("viewCount"),
        "likeCount": d.get("likeCount"),
        "scrapCount": d.get("scrapCount"),
        "shareCount": d.get("shareCount"),
        "commentAndReplyCount": d.get("commentAndReplyCount"),
        "isScrapped": d.get("isScrapped"),
        "isLiked": d.get("isLiked"),

        "writer_id": writer.get("id"),
        "writer_nickname": writer.get("nickname"),
        "writer_profileImage": writer.get("profileImage"),
        "writer_isFollowing": writer.get("isFollowing"),
        "writer_introduction": writer.get("introduction"),
        "writer_userableType": writer.get("userableType"),

        "features_residence": features.get("residence"),
        "features_area": features.get("area"),
        "features_agent": features.get("agent"),
        "features_expertise": features.get("expertise"),
        "features_region": features.get("region"),
        "features_period": features.get("period"),
        "features_budget": features.get("budget"),
        "features_expertBudget": features.get("expertBudget"),
        "features_periodType": features.get("periodType"),

        "features_familyList": json.dumps(features.get("familyList", []), ensure_ascii=False),
        "features_styleList": json.dumps(features.get("styleList", []), ensure_ascii=False),
        "features_constructions": json.dumps(features.get("constructions", []), ensure_ascii=False),
    }

    return post_row


# =========================
# 4. 이미지별 상품태그 추출
# =========================

def parse_image_products(data, content_id):
    d = data.get("data", {})

    title = d.get("title")
    contents = (
        d.get("bpdDocument", {})
        .get("contents", [])
    )

    product_rows = []

    for content_idx, content in enumerate(contents):
        if content.get("type") != "image":
            continue

        image_url = content.get("imageUrl")
        card_id = content.get("extCardId")
        ext_place = content.get("extPlace")
        width = content.get("width")
        height = content.get("height")
        tags = content.get("extTags", [])

        for tag_idx, tag in enumerate(tags):
            production = tag.get("production") or {}

            product_id = tag.get("productionId") or production.get("id")

            brand = tag.get("brand")
            if not brand and isinstance(production.get("brand"), dict):
                brand = production["brand"].get("name")

            name = tag.get("name") or production.get("name")

            product_rows.append({
                "contentId": content_id,
                "postTitle": title,
                "postUrl": f"https://contents.ohou.se/projects/{content_id}",

                "contentIndex": content_idx,
                "imageUrl": image_url,
                "imageWidth": width,
                "imageHeight": height,
                "cardId": card_id,
                "place": ext_place,

                "tagIndex": tag_idx,
                "positionX": tag.get("positionX"),
                "positionY": tag.get("positionY"),
                "description": tag.get("description"),
                "onHide": tag.get("onHide"),
                "isLikely": tag.get("isLikely"),

                "productId": product_id,
                "brand": brand,
                "productName": name,

                "production_id": production.get("id"),
                "production_brand_id": production.get("brand", {}).get("id") if isinstance(production.get("brand"), dict) else None,
                "production_brand_name": production.get("brand", {}).get("name") if isinstance(production.get("brand"), dict) else None,
                "production_name": production.get("name"),
                "originalImageUrl": production.get("originalImageUrl"),
                "originalPrice": production.get("originalPrice"),
                "sellingPrice": production.get("sellingPrice"),

                "productUrl": f"https://ohou.se/productions/{product_id}/selling" if product_id else None
            })

    return product_rows


# =========================
# 5. 여러 게시글 수집 -> 이거는 없어도 되는 함수라고 함
# =========================

def collect_projects(content_ids, save_prefix="housewarming_test"):
    session, headers = make_ohouse_session()

    post_rows = []
    product_rows = []

    for i, content_id in enumerate(content_ids):
        print(f"\n[{i+1}/{len(content_ids)}] 수집 중: {content_id}")

        data = fetch_project_api(content_id, session, headers)

        if data is None:
            continue

        post_row = parse_post_info(data, content_id)
        products = parse_image_products(data, content_id)

        post_rows.append(post_row)
        product_rows.extend(products)

        print(f"✅ 게시글 수집 완료 / 상품태그 {len(products)}개")

        time.sleep(random.uniform(0.8, 1.5))

    posts_df = pd.DataFrame(post_rows)
    image_products_df = pd.DataFrame(product_rows)

    posts_path = f"{save_prefix}_posts.csv"
    products_path = f"{save_prefix}_image_products.csv"

    posts_df.to_csv(posts_path, index=False, encoding="utf-8-sig")
    image_products_df.to_csv(products_path, index=False, encoding="utf-8-sig")

    print("\n저장 완료")
    print(posts_path)
    print(products_path)

    return posts_df, image_products_df

In [2]:
import os
import time
import random
import pandas as pd

def collect_project_ids_from_list_api_resume(
    session,
    headers,
    start_page=1,
    max_pages=900,
    per=20,
    save_path="housewarming_urls.csv"
):
    if os.path.exists(save_path):
        old_df = pd.read_csv(save_path)
        rows = old_df.to_dict("records")
        done_ids = set(old_df["contentId"].astype(int))
        print(f"기존 파일 있음: {len(old_df)}건에서 이어서 수집")
    else:
        rows = []
        done_ids = set()
        print("기존 파일 없음: 처음부터 새로 수집 시작")

    for page in range(start_page, max_pages + 1):
        url = f"https://contents.ohou.se/api/projects?page={page}&per={per}&v=3"

        res = session.get(url, headers=headers, timeout=15)

        if res.status_code != 200:
            print(f"❌ page {page} 실패: {res.status_code}")
            break

        data = res.json()
        projects = data.get("data", {}).get("projects", [])
        next_exists = data.get("data", {}).get("next", False)

        print(f"page {page} / 게시글 {len(projects)}개 / next={next_exists}")

        if len(projects) == 0:
            break

        for p in projects:
            cid = p.get("id")

            if cid in done_ids:
                continue

            rows.append({
                "contentId": cid,
                "url": f"https://contents.ohou.se/projects/{cid}",
                "title": p.get("title"),
                "createdAt": p.get("createdAt"),
                "userId": p.get("userId"),
                "nickname": p.get("nickname"),
                "viewCount": p.get("viewCount"),
                "scrapCount": p.get("scrapCount"),
                "isPro": p.get("isPro"),
                "rating": p.get("rating"),
                "coverImageUrl": p.get("coverImageUrl")
            })

            done_ids.add(cid)

        if page % 20 == 0:
            temp_df = pd.DataFrame(rows).drop_duplicates(subset=["contentId"])
            temp_df.to_csv(save_path, index=False, encoding="utf-8-sig")
            print(f"💾 중간 저장: {len(temp_df)}건")

        if not next_exists:
            break

        time.sleep(random.uniform(0.5, 1.2))

    urls_df = pd.DataFrame(rows).drop_duplicates(subset=["contentId"])
    urls_df.to_csv(save_path, index=False, encoding="utf-8-sig")

    print("최종 저장 완료")
    print("총 게시글 수:", len(urls_df))

    return urls_df

In [3]:
session, headers = make_ohouse_session()

In [ ]:
urls_df = collect_project_ids_from_list_api_resume(
    session=session,
    headers=headers,
    start_page=1,
    max_pages=900,
    per=20,
    save_path="housewarming_urls.csv"
)

2. 게시글 기본 정보 수집 함수

In [4]:
import time
import json
import random
import requests
import pandas as pd


# =========================
# 1. 상세 API 호출
# =========================

def fetch_project_detail(content_id, session, headers):

    url = f"https://contents.ohou.se/api/projects/{content_id}"

    try:
        res = session.get(url, headers=headers, timeout=15)

        if res.status_code != 200:
            print(f"❌ {content_id} 실패: {res.status_code}")
            return None

        return res.json()

    except Exception as e:
        print(f"❌ {content_id} 에러:", e)
        return None


# =========================
# 2. 게시글 기본정보 파싱
# =========================

def parse_post_info(data, content_id):

    d = data.get("data", {}) or {}
    writer = d.get("writer", {}) or {}
    features = d.get("features", {}) or {}

    row = {

        # 기본
        "contentId": d.get("contentId"),
        "contentType": d.get("contentType"),
        "title": d.get("title"),
        "createdAt": d.get("createdAt"),
        "updatedAt": d.get("updatedAt"),

        "url": f"https://contents.ohou.se/projects/{content_id}",
        "coverImageUrl": d.get("coverImageUrl"),

        # 통계
        "viewCount": d.get("viewCount"),
        "likeCount": d.get("likeCount"),
        "scrapCount": d.get("scrapCount"),
        "shareCount": d.get("shareCount"),
        "commentAndReplyCount": d.get("commentAndReplyCount"),

        # 작성자
        "writer_id": writer.get("id"),
        "writer_nickname": writer.get("nickname"),
        "writer_profileImage": writer.get("profileImage"),
        "writer_isFollowing": writer.get("isFollowing"),
        "writer_introduction": writer.get("introduction"),
        "writer_userableType": writer.get("userableType"),

        # 집 정보
        "features_residence": features.get("residence"),
        "features_area": features.get("area"),
        "features_agent": features.get("agent"),
        "features_expertise": features.get("expertise"),
        "features_region": features.get("region"),
        "features_period": features.get("period"),
        "features_budget": features.get("budget"),

        # 리스트형
        "features_familyList": json.dumps(
            features.get("familyList", []),
            ensure_ascii=False
        ),

        "features_styleList": json.dumps(
            features.get("styleList", []),
            ensure_ascii=False
        ),

        "features_constructions": json.dumps(
            features.get("constructions", []),
            ensure_ascii=False
        )
    }

    return row


# =========================
# 3. 전체 게시글 기본정보 수집
# =========================

def collect_post_details(
    content_ids,
    session,
    headers,
    save_path="housewarming_posts_all.csv",
    checkpoint_every=100
):

    rows = []

    total = len(content_ids)

    for idx, content_id in enumerate(content_ids, start=1):

        print(f"[{idx}/{total}] 수집 중: {content_id}")

        data = fetch_project_detail(
            content_id,
            session,
            headers
        )

        if data is None:
            continue

        row = parse_post_info(
            data,
            content_id
        )

        rows.append(row)

        print(f"✅ 성공: {row['title']}")

        # 중간 저장
        if idx % checkpoint_every == 0:

            temp_df = pd.DataFrame(rows)

            temp_df.to_csv(
                save_path,
                index=False,
                encoding="utf-8-sig"
            )

            print(f"💾 중간 저장 완료: {len(temp_df)}건")

        time.sleep(random.uniform(0.5, 1.2))

    posts_df = pd.DataFrame(rows)

    posts_df.to_csv(
        save_path,
        index=False,
        encoding="utf-8-sig"
    )

    print("최종 저장 완료")
    print("총 게시글 수:", len(posts_df))

    return posts_df

In [ ]:
urls_df = pd.read_csv("housewarming_urls.csv")

In [ ]:
content_ids = urls_df["contentId"].tolist()

len(content_ids)

In [ ]:
session, headers = make_ohouse_session()

In [ ]:
posts_df = collect_post_details(
    content_ids=content_ids,
    session=session,
    headers=headers,
    save_path="housewarming_posts_all.csv",
    checkpoint_every=100
)

3. 본문 텍스트 수집 함수

In [5]:
import time
import random
import requests
import pandas as pd


# =========================
# 본문 텍스트 추출
# =========================

def extract_post_text(data):

    d = data.get("data", {}) or {}

    contents = (
        d.get("bpdDocument", {})
        .get("contents", [])
    )

    text_list = []

    for c in contents:

        c_type = c.get("type")

        # 텍스트 계열 블록
        if c_type in ["p", "h1", "h2", "h3", "blockquote", "callout"]:

            text_items = c.get("text", [])

            # callout 대응
            if c_type == "callout":
                text_items = (
                    c.get("titleText", [])
                    + c.get("text", [])
                )

            for item in text_items:

                content = item.get("content", [])

                for x in content:

                    if isinstance(x, str):
                        text_list.append(x)

                    elif isinstance(x, dict):

                        if x.get("type") == "br":
                            text_list.append("\n")

        # 이미지 캡션도 추가
        if c_type == "image":

            caption_items = c.get("caption", [])

            for item in caption_items:

                content = item.get("content", [])

                for x in content:

                    if isinstance(x, str):
                        text_list.append(x)

    full_text = " ".join(text_list)
    full_text = " ".join(full_text.split())

    return full_text


# =========================
# 상세 API 호출
# =========================

def fetch_project_detail(content_id, session, headers):

    url = f"https://contents.ohou.se/api/projects/{content_id}"

    try:

        res = session.get(
            url,
            headers=headers,
            timeout=15
        )

        if res.status_code != 200:
            print(f"❌ {content_id} 실패: {res.status_code}")
            return None

        return res.json()

    except Exception as e:

        print(f"❌ {content_id} 에러:", e)
        return None


# =========================
# 본문 텍스트 수집
# =========================

def collect_post_texts(
    content_ids,
    session,
    headers,
    save_path="housewarming_post_texts.csv",
    checkpoint_every=100
):

    rows = []

    total = len(content_ids)

    for idx, content_id in enumerate(content_ids, start=1):

        print(f"[{idx}/{total}] 본문 수집 중: {content_id}")

        data = fetch_project_detail(
            content_id,
            session,
            headers
        )

        if data is None:
            continue

        post_text = extract_post_text(data)

        rows.append({
            "contentId": content_id,
            "postText": post_text
        })

        print(f"✅ 본문 길이: {len(post_text)}")

        # 중간 저장
        if idx % checkpoint_every == 0:

            temp_df = pd.DataFrame(rows)

            temp_df.to_csv(
                save_path,
                index=False,
                encoding="utf-8-sig"
            )

            print(f"💾 중간 저장 완료: {len(temp_df)}건")

        time.sleep(random.uniform(0.15, 0.35))

    text_df = pd.DataFrame(rows)

    text_df.to_csv(
        save_path,
        index=False,
        encoding="utf-8-sig"
    )

    print("최종 저장 완료")
    print("총 게시글 수:", len(text_df))

    return text_df




In [ ]:
# 실행
# =========================

urls_df = pd.read_csv("housewarming_urls.csv")

content_ids = urls_df["contentId"].tolist()

session, headers = make_ohouse_session()

text_df = collect_post_texts(
    content_ids=content_ids,
    session=session,
    headers=headers,
    save_path="housewarming_post_texts.csv",
    checkpoint_every=100
)

display(text_df.head())

4. 상품 태그 수집

In [6]:
import time
import random
import pandas as pd
from pathlib import Path


# =========================
# 1. 상세 API 호출
# =========================

def fetch_project_detail(content_id, session, headers):
    url = f"https://contents.ohou.se/api/projects/{content_id}"

    try:
        res = session.get(url, headers=headers, timeout=20)

        if res.status_code != 200:
            print(f"❌ {content_id} 실패: {res.status_code}")
            return None

        return res.json()

    except Exception as e:
        print(f"❌ {content_id} 에러:", e)
        return None


# =========================
# 2. 이미지별 상품태그 파싱
# =========================

def parse_image_products(data, content_id):
    d = data.get("data", {}) or {}
    title = d.get("title")

    contents = (
        d.get("bpdDocument", {})
        .get("contents", [])
    )

    product_rows = []

    for content_idx, content in enumerate(contents):
        if content.get("type") != "image":
            continue

        image_url = content.get("imageUrl")
        card_id = content.get("extCardId")
        place = content.get("extPlace")
        width = content.get("width")
        height = content.get("height")
        tags = content.get("extTags", [])

        for tag_idx, tag in enumerate(tags):
            production = tag.get("production") or {}

            product_id = tag.get("productionId") or production.get("id")

            brand = tag.get("brand")
            if not brand and isinstance(production.get("brand"), dict):
                brand = production["brand"].get("name")

            product_name = tag.get("name") or production.get("name")

            product_rows.append({
                # 게시글 정보
                "contentId": content_id,
                "postTitle": title,
                "postUrl": f"https://contents.ohou.se/projects/{content_id}",

                # 이미지 정보
                "contentIndex": content_idx,
                "cardId": card_id,
                "place": place,
                "imageUrl": image_url,
                "imageWidth": width,
                "imageHeight": height,

                # 태그 정보
                "tagIndex": tag_idx,
                "positionX": tag.get("positionX"),
                "positionY": tag.get("positionY"),
                "description": tag.get("description"),
                "onHide": tag.get("onHide"),
                "isLikely": tag.get("isLikely"),

                # 상품 정보
                "productId": product_id,
                "brand": brand,
                "productName": product_name,
                "productUrl": f"https://ohou.se/productions/{product_id}/selling" if product_id else None,

                # production 내부 정보
                "production_id": production.get("id"),
                "production_brand_id": production.get("brand", {}).get("id") if isinstance(production.get("brand"), dict) else None,
                "production_brand_name": production.get("brand", {}).get("name") if isinstance(production.get("brand"), dict) else None,
                "production_name": production.get("name"),
                "originalImageUrl": production.get("originalImageUrl"),
                "originalPrice": production.get("originalPrice"),
                "sellingPrice": production.get("sellingPrice"),
            })

    return product_rows


# =========================
# 3. 상품태그 전체 수집
# =========================

def collect_image_products_all(
    content_ids,
    session,
    headers,
    save_path="housewarming_image_products_all.csv",
    checkpoint_path="housewarming_image_products_checkpoint.csv",
    checkpoint_every=100
):
    save_path = Path(save_path)
    checkpoint_path = Path(checkpoint_path)

    # 이어서 진행 가능
    if checkpoint_path.exists():
        old_df = pd.read_csv(checkpoint_path)

        all_rows = old_df.to_dict("records")

        done_ids = set(
            old_df["contentId"]
            .dropna()
            .astype(int)
            .unique()
        )

        print(f"이어서 진행: 기존 완료 게시글 {len(done_ids)}개 / 기존 행 {len(old_df)}개")

    else:
        all_rows = []
        done_ids = set()

    total = len(content_ids)

    for idx, content_id in enumerate(content_ids, start=1):
        content_id = int(content_id)

        if content_id in done_ids:
            continue

        print(f"[{idx}/{total}] 상품태그 수집 중: {content_id}")

        data = fetch_project_detail(content_id, session, headers)

        if data is None:
            continue

        product_rows = parse_image_products(data, content_id)

        all_rows.extend(product_rows)
        done_ids.add(content_id)

        print(f"✅ 상품태그 {len(product_rows)}개")

        # 중간 저장
        if len(done_ids) % checkpoint_every == 0:
            temp_df = pd.DataFrame(all_rows)

            temp_df.to_csv(
                checkpoint_path,
                index=False,
                encoding="utf-8-sig"
            )

            temp_df.to_csv(
                save_path,
                index=False,
                encoding="utf-8-sig"
            )

            print(f"💾 중간 저장 완료: 게시글 {len(done_ids)}개 / 상품태그 {len(temp_df)}행")

        time.sleep(random.uniform(0.15, 0.35))

    image_products_df = pd.DataFrame(all_rows)

    image_products_df.to_csv(
        checkpoint_path,
        index=False,
        encoding="utf-8-sig"
    )

    image_products_df.to_csv(
        save_path,
        index=False,
        encoding="utf-8-sig"
    )

    print("최종 저장 완료")
    print("완료 게시글 수:", len(done_ids))
    print("상품태그 총 행 수:", len(image_products_df))
    print("저장 파일:", save_path)

    return image_products_df



In [ ]:
# =========================
# 4. 실행
# =========================

urls_df = pd.read_csv("housewarming_urls.csv")

content_ids = (
    urls_df["contentId"]
    .astype(int)
    .tolist()
)

print("총 게시글 수:", len(content_ids))

session, headers = make_ohouse_session()

image_products_df = collect_image_products_all(
    content_ids=content_ids,
    session=session,
    headers=headers,
    save_path="housewarming_image_products_all.csv",
    checkpoint_path="housewarming_image_products_checkpoint.csv",
    checkpoint_every=100
)

print(image_products_df.shape)

display(image_products_df.head())

5. 대표 이미지 rgb 추출

In [7]:
!pip install pillow


[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: C:\Users\SAMSUNG\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [8]:
!pip install scikit-learn


[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: C:\Users\SAMSUNG\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [9]:
import os
import time
import random
import requests
import pandas as pd
import numpy as np

from PIL import Image
from io import BytesIO
from sklearn.cluster import KMeans


def rgb_to_hex(rgb):
    r, g, b = [int(x) for x in rgb]
    return f"#{r:02X}{g:02X}{b:02X}"


def extract_dominant_colors_from_url(
    image_url,
    n_colors=5,
    resize_size=(150, 150),
    timeout=15
):
    try:
        res = requests.get(image_url, timeout=timeout)

        if res.status_code != 200:
            return None

        img = Image.open(BytesIO(res.content)).convert("RGB")
        img = img.resize(resize_size)

        pixels = np.array(img).reshape(-1, 3)

        kmeans = KMeans(
            n_clusters=n_colors,
            random_state=42,
            n_init=10
        )

        kmeans.fit(pixels)

        colors = kmeans.cluster_centers_.astype(int)
        labels = kmeans.labels_

        counts = np.bincount(labels)
        ratios = counts / counts.sum()

        order = np.argsort(counts)[::-1]

        result = []

        for idx in order:
            rgb = colors[idx]
            ratio = ratios[idx]

            result.append({
                "r": int(rgb[0]),
                "g": int(rgb[1]),
                "b": int(rgb[2]),
                "hex": rgb_to_hex(rgb),
                "ratio": round(float(ratio), 4)
            })

        return result

    except Exception as e:
        return None


def collect_cover_image_colors(
    posts_csv_path="housewarming_posts_final.csv",
    save_path="housewarming_cover_image_colors.csv",
    n_colors=5,
    checkpoint_every=100
):
    posts_df = pd.read_csv(posts_csv_path)

    if os.path.exists(save_path):
        old_df = pd.read_csv(save_path)
        done_ids = set(old_df["contentId"].astype(int))
        rows = old_df.to_dict("records")
        print(f"기존 파일 있음: {len(old_df)}건에서 이어서 시작")
    else:
        done_ids = set()
        rows = []
        print("기존 파일 없음: 처음부터 시작")

    total = len(posts_df)

    for idx, row in posts_df.iterrows():
        content_id = int(row["contentId"])
        image_url = row["coverImageUrl"]

        if content_id in done_ids:
            continue

        print(f"[{idx + 1}/{total}] 대표색 추출 중: {content_id}")

        if pd.isna(image_url):
            print("❌ 이미지 URL 없음")
            continue

        colors = extract_dominant_colors_from_url(
            image_url=image_url,
            n_colors=n_colors
        )

        if colors is None:
            print("❌ 대표색 추출 실패")
            continue

        result_row = {
            "contentId": content_id,
            "coverImageUrl": image_url
        }

        for i, color in enumerate(colors, start=1):
            result_row[f"color{i}_r"] = color["r"]
            result_row[f"color{i}_g"] = color["g"]
            result_row[f"color{i}_b"] = color["b"]
            result_row[f"color{i}_hex"] = color["hex"]
            result_row[f"color{i}_ratio"] = color["ratio"]

        rows.append(result_row)
        done_ids.add(content_id)

        print(f"✅ 완료: {colors[0]['hex']}")

        if len(rows) % checkpoint_every == 0:
            temp_df = pd.DataFrame(rows)
            temp_df.to_csv(save_path, index=False, encoding="utf-8-sig")
            print(f"💾 중간 저장 완료: {len(temp_df)}건")

        time.sleep(random.uniform(0.2, 0.5))

    colors_df = pd.DataFrame(rows)
    colors_df.to_csv(save_path, index=False, encoding="utf-8-sig")

    print("최종 저장 완료")
    print("총 대표색 추출 게시글 수:", len(colors_df))

    return colors_df

In [10]:
colors_df = collect_cover_image_colors(
    posts_csv_path="housewarming_posts_final.csv",
    save_path="housewarming_cover_image_colors.csv",
    n_colors=5,
    checkpoint_every=100
)

기존 파일 없음: 처음부터 시작
[1/8000] 대표색 추출 중: 193499
✅ 완료: #4A4946
[2/8000] 대표색 추출 중: 185117
✅ 완료: #B8B7B5
[3/8000] 대표색 추출 중: 183328
✅ 완료: #A1A09D
[4/8000] 대표색 추출 중: 195099
✅ 완료: #CAC5BB
[5/8000] 대표색 추출 중: 194221
✅ 완료: #75665B
[6/8000] 대표색 추출 중: 193576
✅ 완료: #A19F99
[7/8000] 대표색 추출 중: 184452
✅ 완료: #A0A9A9
[8/8000] 대표색 추출 중: 178429
✅ 완료: #A8A5A2
[9/8000] 대표색 추출 중: 194303
✅ 완료: #241815
[10/8000] 대표색 추출 중: 193785
✅ 완료: #ABA59D
[11/8000] 대표색 추출 중: 173927
✅ 완료: #A7A19B
[12/8000] 대표색 추출 중: 152380
✅ 완료: #CFCBCC
[13/8000] 대표색 추출 중: 194636
✅ 완료: #BC9E86
[14/8000] 대표색 추출 중: 193496
✅ 완료: #B08765
[15/8000] 대표색 추출 중: 193484
✅ 완료: #9C988D
[16/8000] 대표색 추출 중: 193524
✅ 완료: #9C9585
[17/8000] 대표색 추출 중: 193277
✅ 완료: #BFCBC9
[18/8000] 대표색 추출 중: 186446
✅ 완료: #B89E88
[19/8000] 대표색 추출 중: 185262
✅ 완료: #B7B8B3
[20/8000] 대표색 추출 중: 177101
✅ 완료: #BDB9B4
[21/8000] 대표색 추출 중: 193129
✅ 완료: #A69D92
[22/8000] 대표색 추출 중: 193057
✅ 완료: #B7B6B0
[23/8000] 대표색 추출 중: 186086
✅ 완료: #D2C6B7
[24/8000] 대표색 추출 중: 149342
✅ 완료: #9C9894
[25/800